# Chapter 10: true COGS per hL and the brand by pack grid

Reads the March cost ledger and the month's volumes by brand and pack, shares overhead three ways and draws the grid. Kestrel's March is a two brand month: lager in bottles and kegs, pale ale in cans.

The ledger books materials by brand and packaging by store: cans come from the can line's own store, bottles and kegs share one. Within that store, bottles and kegs split by their standard cost (36 and 2 USD a hL).

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ledger = pd.read_csv("../data/cost_ledger_march.csv")
brands = pd.read_csv("../data/march_brands.csv").set_index("brand")
packs = pd.read_csv("../data/march_brand_pack.csv")
hl = brands.hl_sold.sum()
cogs = ledger[ledger.in_cogs == "yes"].groupby("line", sort=False).usd_march.sum()
t = cogs.to_frame("USD, March")
t["Per hL sold"] = t["USD, March"] / hl
print(t.round(2))
print(f"COGS before duty USD {cogs.sum():,.0f}, {cogs.sum() / hl:.2f} a hL (INR {cogs.sum() / hl * 90:,.0f}; EUR {cogs.sum() / hl / 1.1:.2f})")
cast = 1.045 * hl
print(f"Divide by hL cast ({cast:,.0f}) instead and it reads {cogs.sum() / cast:.2f}: a gap of {cogs.sum() / hl - cogs.sum() / cast:.2f}")

## Overhead three ways (chapter 10.2)

In [ ]:
pool = cogs["Overhead"]
brands["hl_days"] = brands.hl_sold * brands.tank_days_a_fill
alloc = pd.DataFrame({
    "Volume": pool / hl,
    "Brew": pool / brands.brews.sum() * brands.brews / brands.hl_sold,
    "Tank day": pool / brands.hl_days.sum() * brands.tank_days_a_fill,
}, index=brands.index)
print(f"Per brew {pool / brands.brews.sum():,.2f}; per hL-day {pool / brands.hl_days.sum():.4f}")
print(alloc.round(2))
print("Each basis adds back to the pool:", (alloc.mul(brands.hl_sold, axis=0).sum().round(0) == pool).all())

## The brand by pack grid (chapters 10.3 and 10.5)

In [ ]:
def ledger_usd(line, centre):
    return ledger.loc[(ledger.line == line) & (ledger.cost_centre == centre), "usd_march"].sum()


labour_utilities = (cogs["Direct labour"] + cogs["Utilities"]) / hl
materials = {b: ledger_usd("Materials", b) / brands.loc[b, "hl_sold"] for b in brands.index}
bk = packs[packs.pack.isin(["Bottle", "Keg"])]
bk_share = bk.hl_sold * bk.packaging_standard_usd_hl
packaging = dict(zip(bk.pack, ledger_usd("Packaging", "Bottle and keg store") * bk_share / bk_share.sum() / bk.hl_sold))
packaging["Can"] = ledger_usd("Packaging", "Can line") / packs.loc[packs.pack == "Can", "hl_sold"].iloc[0]
packs["true_cost_usd_hl"] = [materials[b] + packaging[p] + labour_utilities + alloc.loc[b, "Tank day"] for b, p in zip(packs.brand, packs.pack)]
grid = packs.pivot(index="brand", columns="pack", values="true_cost_usd_hl").round(2)
print(grid.fillna("none this month"))

packs["usd"] = packs.true_cost_usd_hl * packs.hl_sold
brand_cost = packs.groupby("brand").usd.sum() / brands.hl_sold
print("Brand true cost:", brand_cost.round(2).to_dict(), "Plant:", round(packs.usd.sum() / hl, 2))

In [ ]:
direct = brand_cost - alloc["Tank day"]
w = pd.DataFrame({
    "Cost on paper": cogs.sum() / hl,
    "Margin on paper": brands.net_price_usd_hl - cogs.sum() / hl,
    "Direct cost": direct,
    "Overhead by tank day": alloc["Tank day"],
    "True cost": brand_cost,
    "True margin": brands.net_price_usd_hl - brand_cost,
}).T
w.loc["March margin, USD"] = w.loc["True margin"] * brands.hl_sold
print(w.round(2))
print(f"Total margin {w.loc['March margin, USD'].sum():,.0f}; lager true cost INR {brand_cost['Lager'] * 90:,.0f}, EUR {brand_cost['Lager'] / 1.1:.2f}")
print(f"Ale against lager per hL: on paper {w.loc['Margin on paper', 'Pale ale'] / w.loc['Margin on paper', 'Lager']:.2f}, true {w.loc['True margin', 'Pale ale'] / w.loc['True margin', 'Lager']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.6))
g = packs.pivot(index="brand", columns="pack", values="true_cost_usd_hl")[["Bottle", "Keg", "Can"]]
ax.imshow(g.fillna(0).values, cmap="Greens", vmin=0, vmax=120)
for i in range(g.shape[0]):
    for j in range(g.shape[1]):
        v = g.iloc[i, j]
        ax.text(j, i, "none this month" if pd.isna(v) else f"{v:.2f}", ha="center", va="center")
ax.set_xticks(range(3), g.columns)
ax.set_yticks(range(2), g.index)
ax.set_title("Figure 10.2: true cost, USD a hL, March (synthetic)")
plt.tight_layout()
plt.show()